In [ ]:
# import the required libraries
import pandas as pd
from transformers import GPT2Tokenizer, GPT2Model
import torch
import numpy as np

# read the input data (cleaned embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title dataset is the dataset 
formed by merging & cleaning the embedded keywords and embedded title dataset.
"""
movies_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/gpt2_chunk_5.csv")

In [ ]:
movies_df.head(10)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,866558,"Documentary, Music",NaN,ko,BTS Memories of 2020,BTS Memories of 2020 DVD depicts glorious mome...,1.414,8/3/2021,Korean,10.0,1.0,866558.0,behind the scenes k-pop bts,"[11, 18]",-3.47666383e-01 4.13006544e-02 -3.24344546e-0...,1.40299857e-01 -5.95012167e-03 -9.35773626e-0...
1,False,866560,"Horror, Music",NaN,de,"ig,gz,",NaN,0.600,11/24/2003,[],0.0,0.0,866560.0,baseball dachboden,"[12, 18]",-8.89021680e-02 -2.59780318e-01 -3.78739506e-0...,-3.26778650e-01 -7.42522180e-02 -1.58985019e-0...
2,False,866567,Drama,tt8380392,ja,é¯‰ã®ã¯ãªã‚·ã‚¢ã‚¿ãƒ¼,NaN,1.400,9/7/2018,Japanese,3.5,1.0,866567.0,baseball,[8],4.37184125e-02 -2.60913879e-01 -1.68581203e-0...,1.60534456e-01 -1.59905672e-01 3.15829873e-0...
3,False,866582,Music,NaN,en,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,0.841,12/31/2000,[],0.0,0.0,866582.0,new year's eve berlin germany concert,[18],1.74181968e-01 -4.89853546e-02 -3.20719272e-0...,-1.48498401e-01 1.23102792e-01 2.82955885e-0...
4,False,866583,"Documentary, Music",NaN,pt,Sexo Ã© Bom,NaN,0.664,1/1/1987,Portuguese,0.0,0.0,866583.0,sexuality pop singer pop music brazilian music...,"[11, 18]",-1.73860773e-01 4.93408293e-01 -3.40111107e-0...,-9.35127512e-02 -5.31892002e-01 2.60485649e-0...
5,False,866586,Music,NaN,en,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,1.877,12/31/2001,[],0.0,0.0,866586.0,new year's eve berlin germany concert,[18],1.74181968e-01 -4.89853546e-02 -3.20719272e-0...,-1.39027864e-01 1.04881302e-01 2.59337485e-0...
6,False,866587,Comedy,tt11412672,da,Conrad Molden - Danglish 2.0,"British comedian Conrad Molden is ready with, ...",0.600,9/2/2021,"Danish, English",8.0,1.0,866587.0,stand-up comedy,[5],-6.52225763e-02 1.68742985e-02 -3.78949761e-0...,-6.24902509e-02 -2.95983434e-01 -6.61591291e-0...
7,False,866589,Music,NaN,en,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,1.576,12/31/2002,[],0.0,0.0,866589.0,concert,[18],-2.41739914e-01 -1.25383609e-04 -3.48371297e-0...,-1.30962163e-01 1.01863839e-01 2.59276122e-0...
8,False,866591,Music,NaN,en,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,1.690,12/31/2004,[],0.0,0.0,866591.0,concert,[18],-2.41739914e-01 -1.25383609e-04 -3.48371297e-0...,-1.29500657e-01 1.05551496e-01 2.56707370e-0...
9,False,866595,Music,NaN,en,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,1.400,12/31/2007,[],0.0,0.0,866595.0,concert,[18],-2.41739914e-01 -1.25383609e-04 -3.48371297e-0...,-1.30178764e-01 1.05597086e-01 2.53991932e-0...


In [ ]:
movies_df.shape[0]

46943

In [ ]:
movies_df.columns

Index(['adult', 'id', 'genres', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
drop_columns = ['adult', 'genres', 'original_language',
       'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview
0,866558,NaN,BTS Memories of 2020,BTS Memories of 2020 DVD depicts glorious mome...
1,866560,NaN,"ig,gz,",NaN
2,866567,tt8380392,é¯‰ã®ã¯ãªã‚·ã‚¢ã‚¿ãƒ¼,NaN
3,866582,NaN,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...
4,866583,NaN,Sexo Ã© Bom,NaN


In [ ]:
# reduce the size of the overview column to 500 characters of string
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column '{}': {}".format('overview', max_length))
movies_df['overview'] = movies_df['overview'].str.slice(0, 500)
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column after strip '{}': {}".format('overview', max_length))

Maximum length of column 'overview': 1085.0
Maximum length of column after strip 'overview': 500.0


In [ ]:
# load the pre-trained gpt-2 tokenizer and model
model_name='gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2Model.from_pretrained(model_name)

def create_word_embeddings(sentence):
    if pd.isna(sentence):
      return np.zeros(768)
    else:
      input_ids = tokenizer.encode(sentence, return_tensors='pt')
      with torch.no_grad():
          output = model(input_ids)
      embeddings = output.last_hidden_state.mean(dim=1).squeeze().numpy()
      return embeddings

# Apply the function to each row and create a new column 'overview_embedding'
movies_df['overview_embedding'] = movies_df['overview'].apply(create_word_embeddings)

In [ ]:
movies_df['overview_embedding'] = movies_df['overview_embedding'].apply(lambda x: str(x))
movies_df.loc[0, 'overview_embedding']

'[-6.60052896e-02 -5.21661229e-02 -2.03125224e-01  2.17369244e-01\n -1.46773323e-01 -5.93923628e-02  5.19420910e+00 -5.17166555e-02\n  2.30836973e-01 -3.01144689e-01 -1.20326793e-02 -2.73235496e-02\n -3.92289720e-02 -3.54325958e-02  3.99986422e-03  1.56024620e-01\n  1.75770923e-01  6.42171323e-01  7.46513307e-02 -2.56520659e-01\n -1.32357806e-01 -1.58611298e-01 -3.52727585e-02  1.53161004e-01\n -1.49791390e-02  3.19585279e-02 -4.44573760e-01 -2.22499788e-01\n -3.51245880e-01  1.33758470e-01 -1.40436709e-01 -1.85119703e-01\n  1.95496306e-01 -1.77833304e-01 -2.67068356e-01 -4.18965667e-01\n  7.01835709e+01 -1.94399431e-01 -2.42136955e-01  4.34848577e-01\n -1.13570146e-01 -8.01959708e-02  2.90013850e-03 -3.17934193e-02\n -1.98865071e-01  1.35885954e-01  1.79725755e-02  9.44477096e-02\n  1.08488329e-01  1.51527718e-01 -7.69182667e-02 -8.44790265e-02\n -2.35979319e-01 -8.32354352e-02  3.94541025e-01  8.00722837e-01\n  2.83469290e-01  2.63426542e-01 -1.52695343e-01  6.36727884e-02\n -1.57610

In [ ]:
# convert the embedded column array to embedded string and clean the overview_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_df['overview_embedding_string'] = movies_df['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '').str.replace(',', '')

In [ ]:
movies_df.loc[0, 'overview_embedding_string']

'-6.60052896e-02 -5.21661229e-02 -2.03125224e-01  2.17369244e-01 -1.46773323e-01 -5.93923628e-02  5.19420910e+00 -5.17166555e-02  2.30836973e-01 -3.01144689e-01 -1.20326793e-02 -2.73235496e-02 -3.92289720e-02 -3.54325958e-02  3.99986422e-03  1.56024620e-01  1.75770923e-01  6.42171323e-01  7.46513307e-02 -2.56520659e-01 -1.32357806e-01 -1.58611298e-01 -3.52727585e-02  1.53161004e-01 -1.49791390e-02  3.19585279e-02 -4.44573760e-01 -2.22499788e-01 -3.51245880e-01  1.33758470e-01 -1.40436709e-01 -1.85119703e-01  1.95496306e-01 -1.77833304e-01 -2.67068356e-01 -4.18965667e-01  7.01835709e+01 -1.94399431e-01 -2.42136955e-01  4.34848577e-01 -1.13570146e-01 -8.01959708e-02  2.90013850e-03 -3.17934193e-02 -1.98865071e-01  1.35885954e-01  1.79725755e-02  9.44477096e-02  1.08488329e-01  1.51527718e-01 -7.69182667e-02 -8.44790265e-02 -2.35979319e-01 -8.32354352e-02  3.94541025e-01  8.00722837e-01  2.83469290e-01  2.63426542e-01 -1.52695343e-01  6.36727884e-02 -1.57610446e-01 -9.05631259e-02 -4.4415

In [ ]:
drop_columns = ['overview_embedding']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview,overview_embedding_string
0,866558,NaN,BTS Memories of 2020,BTS Memories of 2020 DVD depicts glorious mome...,-6.60052896e-02 -5.21661229e-02 -2.03125224e-0...
1,866560,NaN,"ig,gz,",NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...
2,866567,tt8380392,é¯‰ã®ã¯ãªã‚·ã‚¢ã‚¿ãƒ¼,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...
3,866582,NaN,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,4.86629829e-02 -1.32747680e-01 -2.26144001e-0...
4,866583,NaN,Sexo Ã© Bom,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...


In [ ]:
movies_df['overview_embedding_string'] = movies_df['overview_embedding_string'].str.strip()

In [ ]:
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/old_final_mkg_dataset_gpt2.csv")

In [ ]:
merged_df = pd.merge(movies_df, movies_keywords_genres, on=['id', 'imdb_id', 'original_title'], how='inner')

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,866558,NaN,BTS Memories of 2020,BTS Memories of 2020 DVD depicts glorious mome...,-6.60052896e-02 -5.21661229e-02 -2.03125224e-0...,False,"Documentary, Music",ko,BTS Memories of 2020 DVD depicts glorious mome...,1.414,8/3/2021,Korean,10.0,1.0,866558.0,behind the scenes k-pop bts,"[11, 18]",-3.47666383e-01 4.13006544e-02 -3.24344546e-0...,1.40299857e-01 -5.95012167e-03 -9.35773626e-0...
1,866560,NaN,"ig,gz,",NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,"Horror, Music",de,NaN,0.600,11/24/2003,[],0.0,0.0,866560.0,baseball dachboden,"[12, 18]",-8.89021680e-02 -2.59780318e-01 -3.78739506e-0...,-3.26778650e-01 -7.42522180e-02 -1.58985019e-0...
2,866567,tt8380392,é¯‰ã®ã¯ãªã‚·ã‚¢ã‚¿ãƒ¼,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Drama,ja,NaN,1.400,9/7/2018,Japanese,3.5,1.0,866567.0,baseball,[8],4.37184125e-02 -2.60913879e-01 -1.68581203e-0...,1.60534456e-01 -1.59905672e-01 3.15829873e-0...
3,866582,NaN,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,4.86629829e-02 -1.32747680e-01 -2.26144001e-01...,False,Music,en,The annual New Yearâ€™s Eve Concert is one of ...,0.841,12/31/2000,[],0.0,0.0,866582.0,new year's eve berlin germany concert,[18],1.74181968e-01 -4.89853546e-02 -3.20719272e-0...,-1.48498401e-01 1.23102792e-01 2.82955885e-0...
4,866583,NaN,Sexo Ã© Bom,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,"Documentary, Music",pt,NaN,0.664,1/1/1987,Portuguese,0.0,0.0,866583.0,sexuality pop singer pop music brazilian music...,"[11, 18]",-1.73860773e-01 4.93408293e-01 -3.40111107e-0...,-9.35127512e-02 -5.31892002e-01 2.60485649e-0...


In [ ]:
merged_df.shape[0]

46943

In [ ]:
merged_df.columns

Index(['id', 'imdb_id', 'original_title', 'overview_x',
       'overview_embedding_string', 'adult', 'genres', 'original_language',
       'overview_y', 'popularity', 'release_date', 'spoken_languages',
       'vote_average', 'vote_count', 'movie_id', 'keywords', 'genres_indices',
       'keywords_embedding_string', 'original_title_embedding_string'],
      dtype='object')

In [ ]:
nan_count = merged_df['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding' has {nan_count} NaN values.")
nan_count = merged_df['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
merged_df.dropna(subset=['overview_embedding_string'], inplace=True)
merged_df.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
merged_df.shape[0]

46943

In [ ]:
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/final_mkg_dataset_gpt2_chunk_5.csv', index=False)

In [ ]:
#  Test

In [ ]:
movies_df.shape[0]

46943

In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,866558,NaN,BTS Memories of 2020,BTS Memories of 2020 DVD depicts glorious mome...,-6.60052896e-02 -5.21661229e-02 -2.03125224e-0...,False,"Documentary, Music",ko,BTS Memories of 2020 DVD depicts glorious mome...,1.414,8/3/2021,Korean,10.0,1.0,866558.0,behind the scenes k-pop bts,"[11, 18]",-3.47666383e-01 4.13006544e-02 -3.24344546e-0...,1.40299857e-01 -5.95012167e-03 -9.35773626e-0...
1,866560,NaN,"ig,gz,",NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,"Horror, Music",de,NaN,0.600,11/24/2003,[],0.0,0.0,866560.0,baseball dachboden,"[12, 18]",-8.89021680e-02 -2.59780318e-01 -3.78739506e-0...,-3.26778650e-01 -7.42522180e-02 -1.58985019e-0...
2,866567,tt8380392,é¯‰ã®ã¯ãªã‚·ã‚¢ã‚¿ãƒ¼,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Drama,ja,NaN,1.400,9/7/2018,Japanese,3.5,1.0,866567.0,baseball,[8],4.37184125e-02 -2.60913879e-01 -1.68581203e-0...,1.60534456e-01 -1.59905672e-01 3.15829873e-0...
3,866582,NaN,The Berliner Philharmonikerâ€™s New Yearâ€™s E...,The annual New Yearâ€™s Eve Concert is one of ...,4.86629829e-02 -1.32747680e-01 -2.26144001e-01...,False,Music,en,The annual New Yearâ€™s Eve Concert is one of ...,0.841,12/31/2000,[],0.0,0.0,866582.0,new year's eve berlin germany concert,[18],1.74181968e-01 -4.89853546e-02 -3.20719272e-0...,-1.48498401e-01 1.23102792e-01 2.82955885e-0...
4,866583,NaN,Sexo Ã© Bom,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,"Documentary, Music",pt,NaN,0.664,1/1/1987,Portuguese,0.0,0.0,866583.0,sexuality pop singer pop music brazilian music...,"[11, 18]",-1.73860773e-01 4.93408293e-01 -3.40111107e-0...,-9.35127512e-02 -5.31892002e-01 2.60485649e-0...


In [ ]:
# merge the embedded keywords, title and overview columns to form the input gpt2 embedded dataset chunk-5
drop_columns = ['imdb_id', 'original_title', 'overview_x', 'adult', 'original_language', 'overview_y', 'popularity', 'spoken_languages', 'vote_average', 'vote_count', 'movie_id', 'keywords']
merged_df = merged_df.drop(columns=drop_columns)
merged_df.head(5)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,866558,-6.60052896e-02 -5.21661229e-02 -2.03125224e-0...,"Documentary, Music",8/3/2021,"[11, 18]",-3.47666383e-01 4.13006544e-02 -3.24344546e-0...,1.40299857e-01 -5.95012167e-03 -9.35773626e-0...
1,866560,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,"Horror, Music",11/24/2003,"[12, 18]",-8.89021680e-02 -2.59780318e-01 -3.78739506e-0...,-3.26778650e-01 -7.42522180e-02 -1.58985019e-0...
2,866567,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Drama,9/7/2018,[8],4.37184125e-02 -2.60913879e-01 -1.68581203e-0...,1.60534456e-01 -1.59905672e-01 3.15829873e-0...
3,866582,4.86629829e-02 -1.32747680e-01 -2.26144001e-01...,Music,12/31/2000,[18],1.74181968e-01 -4.89853546e-02 -3.20719272e-0...,-1.48498401e-01 1.23102792e-01 2.82955885e-0...
4,866583,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,"Documentary, Music",1/1/1987,"[11, 18]",-1.73860773e-01 4.93408293e-01 -3.40111107e-0...,-9.35127512e-02 -5.31892002e-01 2.60485649e-0...


In [ ]:
merged_df.shape[0]

46943

In [ ]:
# export the embedded keywords, title and overview dataset (chunk-5)
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_5.csv', index=False)

In [ ]:
# Testing read

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_5.csv")

In [ ]:
test_df.shape[0]

46943

In [ ]:
test_df.columns

Index(['id', 'overview_embedding_string', 'genres', 'release_date',
       'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
test_df.head(4)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,866558,-6.60052896e-02 -5.21661229e-02 -2.03125224e-0...,"Documentary, Music",8/3/2021,"[11, 18]",-3.47666383e-01 4.13006544e-02 -3.24344546e-0...,1.40299857e-01 -5.95012167e-03 -9.35773626e-0...
1,866560,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,"Horror, Music",11/24/2003,"[12, 18]",-8.89021680e-02 -2.59780318e-01 -3.78739506e-0...,-3.26778650e-01 -7.42522180e-02 -1.58985019e-0...
2,866567,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Drama,9/7/2018,[8],4.37184125e-02 -2.60913879e-01 -1.68581203e-0...,1.60534456e-01 -1.59905672e-01 3.15829873e-0...
3,866582,4.86629829e-02 -1.32747680e-01 -2.26144001e-01...,Music,12/31/2000,[18],1.74181968e-01 -4.89853546e-02 -3.20719272e-0...,-1.48498401e-01 1.23102792e-01 2.82955885e-0...
